In [1]:
# =============================================================
# CELL 1 — SETUP & LOAD
# =============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 110

BASE = "https://raw.githubusercontent.com/wip-0/ds207_final_project/main/data/interim/split_groupshuffle/"

X_train = pd.read_csv(BASE + "X_train_mini.csv", na_values="?", low_memory=False)
y_train = pd.read_csv(BASE + "y_train_mini.csv").squeeze()

# Rejoin for EDA — target already binarized (0=NO, 1=readmitted)
df = pd.concat([X_train, y_train], axis=1)
print(f"Dataset shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
print(f"\nTarget value counts:\n{df['readmitted'].value_counts().to_string()}")

In [2]:
# =============================================================
# CELL 2 — COLUMN OVERVIEW
# =============================================================
print("\n=== COLUMN OVERVIEW ===")
print(df.dtypes.rename("dtype").to_frame()
      .assign(nulls=df.isnull().sum(),
              pct_null=(df.isnull().mean() * 100).round(1))
      .to_string())

In [3]:
# =============================================================
# CELL 3 — MISSING VALUES
# =============================================================
miss = df.isnull().mean().sort_values(ascending=False)
miss = miss[miss > 0]

fig, ax = plt.subplots(figsize=(8, max(3, len(miss) * 0.4)))
miss.mul(100).plot(kind="barh", ax=ax, color="steelblue", edgecolor="white")
ax.set_xlabel("% Missing")
ax.set_title("Missing Value Rate by Column", fontweight="bold")
ax.axvline(50, color="tomato", linestyle="--", linewidth=1.2, label="50% threshold")
ax.legend()
plt.tight_layout()
plt.show()

high_miss = miss[miss > 0.5].index.tolist()
print(f"Columns >50% missing (likely to drop): {high_miss}")

In [4]:
# =============================================================
# CELL 4 — TARGET VARIABLE: readmitted (binarized)
# =============================================================
target_counts = df["readmitted"].value_counts().sort_index()
labels = {0: "Not Readmitted (0)", 1: "Readmitted (1)"}

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
target_counts.plot(kind="bar", ax=axes[0], color=["#4878d0", "#ee854a"],
                   edgecolor="white", width=0.5)
axes[0].set_title("Readmission Counts", fontweight="bold")
axes[0].set_xlabel("readmitted")
axes[0].set_xticklabels(["Not Readmitted (0)", "Readmitted (1)"], rotation=0)
for p in axes[0].patches:
    axes[0].annotate(f"{int(p.get_height()):,}",
                     (p.get_x() + p.get_width() / 2, p.get_height()),
                     ha="center", va="bottom", fontsize=10)

# Pie chart
axes[1].pie(target_counts, labels=["Not Readmitted (0)", "Readmitted (1)"],
            autopct="%1.1f%%", startangle=140,
            colors=["#4878d0", "#ee854a"])
axes[1].set_title("Readmission Distribution", fontweight="bold")

plt.suptitle("Target: readmitted (binary: 0=NO, 1=Readmitted)", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [5]:
# =============================================================
# CELL 5 — PATIENT DEMOGRAPHICS
# =============================================================
age_order = ["[0-10)","[10-20)","[20-30)","[30-40)","[40-50)",
             "[50-60)","[60-70)","[70-80)","[80-90)","[90-100)"]
age_ct = df["age"].value_counts().reindex(age_order)

fig, ax = plt.subplots(figsize=(9, 4))
age_ct.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white", width=0.7)
ax.set_title("Patient Age Distribution (Training Set)", fontweight="bold")
ax.set_xlabel("Age group")
ax.set_ylabel("Count")
ax.set_xticklabels(age_order, rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Race & Gender
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

race_ct = df["race"].value_counts(dropna=False)
race_ct.plot(kind="bar", ax=axes[0], color="teal", edgecolor="white", width=0.6)
axes[0].set_title("Race", fontweight="bold")
axes[0].set_xlabel("Race")
axes[0].set_ylabel("Count")
axes[0].set_xticklabels(race_ct.index, rotation=30, ha="right")

gen_ct = df["gender"].value_counts()
gen_ct.plot(kind="bar", ax=axes[1], color=["#ee854a","#4878d0","gray"],
            edgecolor="white", width=0.5)
axes[1].set_title("Gender", fontweight="bold")
axes[1].set_xlabel("Gender")
axes[1].set_ylabel("Count")
axes[1].set_xticklabels(gen_ct.index, rotation=0)

plt.suptitle("Patient Demographics", fontweight="bold")
plt.tight_layout()
plt.show()

In [6]:
# =============================================================
# CELL 6 — NUMERIC FEATURE DISTRIBUTIONS
# =============================================================
num_cols = ["time_in_hospital", "num_lab_procedures", "num_procedures",
            "num_medications", "number_outpatient", "number_emergency",
            "number_inpatient", "number_diagnoses"]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=30, color="steelblue", edgecolor="white", linewidth=0.4)
    axes[i].set_title(col.replace("_", " ").title(), fontsize=10)
    axes[i].set_xlabel("Value")
    axes[i].set_ylabel("Count")
    mean_val = df[col].mean()
    axes[i].axvline(mean_val, color="tomato", linestyle="--", linewidth=1.2,
                    label=f"mean={mean_val:.1f}")
    axes[i].legend(fontsize=8)

plt.suptitle("Numeric Feature Distributions (Training Set)", fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [7]:
# =============================================================
# CELL 7 — NUMERIC FEATURES vs READMISSION STATUS
# =============================================================
# Target is binary: 0 = Not Readmitted, 1 = Readmitted
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
palette = {0: "#4878d0", 1: "#ee854a"}

for i, col in enumerate(num_cols):
    sns.boxplot(data=df, x="readmitted", y=col,
                palette=palette, ax=axes[i],
                flierprops={"marker": ".", "markersize": 2})
    axes[i].set_title(col.replace("_", " ").title(), fontsize=10)
    axes[i].set_xlabel("Readmitted (0=No, 1=Yes)")
    axes[i].set_ylabel(col.replace("_", " ").title())

plt.suptitle("Numeric Features by Readmission Status", fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [8]:
# =============================================================
# CELL 8 — MEDICATION USAGE OVERVIEW
# =============================================================
med_cols = ["metformin","repaglinide","nateglinide","chlorpropamide","glimepiride",
            "glipizide","glyburide","pioglitazone","rosiglitazone","insulin"]

def active(series):
    return (series != "No").mean() * 100

med_usage = pd.Series({c: active(df[c]) for c in med_cols}).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
med_usage.plot(kind="bar", ax=ax, color="mediumseagreen", edgecolor="white", width=0.6)
ax.set_title("Medication Usage Rate (% encounters with non-'No' value)", fontweight="bold")
ax.set_xlabel("Medication")
ax.set_ylabel("% of encounters")
ax.set_xticklabels(med_usage.index, rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [9]:
# =============================================================
# CELL 9 — A1C RESULT & GLUCOSE SERUM vs READMISSION
# =============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, ["A1Cresult", "max_glu_serum"]):
    ct = df.groupby(col)["readmitted"].value_counts(normalize=True).unstack().fillna(0) * 100
    ct.plot(kind="bar", ax=ax, edgecolor="white", width=0.6,
            color=["#4878d0", "#ee854a"])
    ax.set_title(f"{col} vs Readmission (%)", fontweight="bold")
    ax.set_xlabel(col)
    ax.set_ylabel("% of group")
    ax.set_xticklabels(ct.index, rotation=30, ha="right")
    ax.legend(title="Readmitted", labels=["Not Readmitted (0)", "Readmitted (1)"], fontsize=8)

plt.suptitle("Lab Results vs Readmission Status", fontweight="bold")
plt.tight_layout()
plt.show()

In [10]:
# =============================================================
# CELL 10 — CORRELATION HEATMAP
# =============================================================
corr_df = df[num_cols + ["admission_type_id", "discharge_disposition_id",
                          "admission_source_id", "readmitted"]].corr()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr_df, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            linewidths=0.5, ax=ax, annot_kws={"size": 8})
ax.set_title("Correlation Matrix – Numeric Features & Target", fontweight="bold")
plt.tight_layout()
plt.show()

In [11]:
# =============================================================
# CELL 11 — READMISSION RATE BY AGE GROUP
# =============================================================
readmit_age = (df.groupby("age")["readmitted"]
               .mean() * 100
               ).reindex(age_order)

fig, ax = plt.subplots(figsize=(9, 4))
readmit_age.plot(kind="bar", ax=ax, color="coral", edgecolor="white", width=0.6)
ax.set_title("Readmission Rate by Age Group", fontweight="bold")
ax.set_xlabel("Age Group")
ax.set_ylabel("% Readmitted")
ax.set_xticklabels(age_order, rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [12]:
# =============================================================
# CELL 12 — SUMMARY STATISTICS
# =============================================================
summary = df.describe(include="all").T
print(summary.to_string())
print("\n✅  EDA complete.")